In [92]:
import numpy as np
import pyvista as pv
pv.set_jupyter_backend('trame')

In [100]:
def unit_vec_from_angles(theta, phi):
    return np.array([
        np.sin(theta) * np.cos(phi), 
        np.sin(theta) * np.sin(phi), 
        np.cos(theta)
    ])

def cross_product_matrix(x1, x2, x3):
    return np.array([
        [0*x1, -x3, x2],
        [x3, 0*x2, -x1],
        [-x2, x1, 0*x3]])


def rotation_matrix_about_axis(alpha, x1, x2, x3):
    cp_mat = cross_product_matrix(x1, x2, x3)  # Shape: (3, 3, n)
    cp_mat_sq = np.einsum('ij...,jk...->ik...', cp_mat, cp_mat)
    t2 = np.sin(alpha) * cp_mat
    t3 = (1 - np.cos(alpha)) * cp_mat_sq
    return np.eye(3) + t2 + t3


def rotation_matrix_about_sphere_angles(alpha, theta, phi):
    unit_vec = unit_vec_from_angles(theta, phi)
    return rotation_matrix_about_axis(alpha, *unit_vec)


def theta_hat(theta, phi):
    theta_x = np.cos(theta) * np.cos(phi)
    theta_y = np.cos(theta) * np.sin(phi)
    theta_z = -np.sin(theta)
    return np.array([theta_x, theta_y, theta_z])

def phi_hat(theta, phi):
    phi_x = -np.sin(phi)
    phi_y = np.cos(phi)
    phi_z = 0
    return np.array([phi_x, phi_y, phi_z])

def get_theta_phi_from_unit_vec(x1, x2, x3):
    theta = np.arccos(x3)
    phi = np.arctan2(x2, x1)
    return theta, phi

def get_spherical_basis_from_unit_vec(x1, x2, x3):
    theta, phi = get_theta_phi_from_unit_vec(x1, x2, x3)
    return theta_hat(theta, phi), phi_hat(theta, phi)

In [241]:
pvplt = pv.Plotter()
pvplt.enable_anti_aliasing('ssaa', multi_samples=12)
unit_sphere = pv.Sphere(radius=1.0)
pvplt.add_mesh(unit_sphere, color='grey', style='wireframe', opacity=0.9, show_edges=False)
pvplt.enable_hidden_line_removal(all_renderers=False)

origin = np.array([0, 0, 0])
_obs_lens_points = np.array(
    [[1.2, 0, 0],
     [0, 1.2, 0.5]], dtype=float
)
obs_lens_points = _obs_lens_points.copy()
lensing_pln_normal = np.cross(*obs_lens_points)
obs_ray = pv.Line(origin, obs_lens_points[0])
lens_ray = pv.Line(origin, obs_lens_points[1])
pvplt.add_mesh(obs_ray, color='dodgerblue', line_width=3, render_lines_as_tubes=False)
pvplt.add_mesh(lens_ray, color='black', line_width=3, render_lines_as_tubes=False)

theta = np.pi / 4
rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
image_obs = rot_mat @ obs_lens_points[0]
image_obs /= np.linalg.norm(image_obs)

image_pos = pv.Sphere(radius=0.1, center=image_obs)
pvplt.add_mesh(image_pos, color='orange')
lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2).triangulate()
pvplt.add_mesh(lensing_pln, color='grey', opacity=0.7, show_edges=False)
intersection = lensing_pln.intersection(unit_sphere, split_first=False, split_second=True)[0]
pvplt.add_mesh(intersection, color='orange', opacity=0.5, line_width=3, show_edges=True)

# Compute theta, phi hats
theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
theta_h_rot = rot_mat @ theta_h
phi_h_rot = rot_mat @ phi_h
theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

theta = np.arccos(np.dot(theta_h_rot, theta_prime_h))
pvplt.add_text(fr'misalignment: {theta:.3f} rad', name = 'theta', position='lower_left')

arrow_list = []
for centre, direction, color in [
    (obs_lens_points[0], theta_h, 'grey'),
    (obs_lens_points[0], phi_h, 'grey'),
    (image_obs, theta_prime_h, 'grey'),
    (image_obs, phi_prime_h, 'grey'),
    (image_obs, theta_h_rot, 'red'),
    (image_obs, phi_h_rot, 'red'),
]:
    arrow = pv.Arrow(centre, direction)
    pvplt.add_mesh(arrow, color=color, show_edges=False)
    arrow_list.append(arrow)

def update_objects():
    global obs_lens_points
    global lensing_pln_normal
    global lensing_pln
    global intersection
    global theta
    global image_pos
    global arrow_list
    
    _lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2).triangulate()
    lensing_pln.copy_from(_lensing_pln)
    _intersection = _lensing_pln.intersection(unit_sphere, split_first=False, split_second=True)[0]
    intersection.copy_from(_intersection)
    rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
    image_obs = rot_mat @ obs_lens_points[0]
    image_obs /= np.linalg.norm(image_obs)
    _image_pos = pv.Sphere(radius=0.1, center=image_obs)
    image_pos.copy_from(_image_pos)

    # Compute theta, phi hats
    # vectors.mapper.center = obs_lens_points[0]
    theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
    theta_h_rot = rot_mat @ theta_h
    phi_h_rot = rot_mat @ phi_h
    theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

    for arrow, (centre, direction) in zip(arrow_list, [
        (obs_lens_points[0], theta_h),
        (obs_lens_points[0], phi_h),
        (image_obs, theta_prime_h),
        (image_obs, phi_prime_h),
        (image_obs, theta_h_rot),
        (image_obs, phi_h_rot),
    ]):
        _arrow = pv.Arrow(centre, direction, scale=0.2)
        arrow.copy_from(_arrow)

    psi = np.arccos(np.dot(theta_h_rot, theta_prime_h))
    pvplt.remove_actor('theta')
    pvplt.add_text(fr'misalignment: {psi:.3f} rad', name = 'theta', position='lower_left')
    

def callback(point, i):
    global obs_lens_points
    global lensing_pln
    global lensing_pln_normal

    _obs_lens_points[i] = point
    obs_lens_points = _obs_lens_points.copy()
    norms = np.linalg.norm(_obs_lens_points, axis=1)
    obs_lens_points /= norms[:, None]
    lensing_pln_normal = np.cross(*obs_lens_points)
    lensing_pln_normal /= np.linalg.norm(lensing_pln_normal)

    _obs_ray = pv.Line(origin, _obs_lens_points[0])
    _lens_ray = pv.Line(origin, _obs_lens_points[1])
    obs_ray.copy_from(_obs_ray)
    lens_ray.copy_from(_lens_ray)
    update_objects()

def update_angle(val):
    global theta
    theta = val
    update_objects()

pvplt.add_sphere_widget(callback, center=obs_lens_points, color=['dodgerblue', 'black'], radius=0.07)
pvplt.add_slider_widget(
    callback=update_angle,
    rng=[0, 2 * np.pi],
    value=1,
    title='theta',
    pointa=(0.025, 0.5),
    pointb=(0.25, 0.5),
    style='modern',
)
pvplt.show(interactive_update=True)

Widget(value='<iframe src="http://localhost:65489/index.html?ui=P_0x57be9eca0_148&reconnect=auto" class="pyvis…

## Animation?

In [237]:
pvplt = pv.Plotter(off_screen=True)
pvplt.enable_anti_aliasing('ssaa', multi_samples=32)

unit_sphere = pv.Sphere(radius=1.0)
pvplt.add_mesh(unit_sphere, color='grey', style='wireframe', opacity=0.9, show_edges=False)
pvplt.enable_hidden_line_removal(all_renderers=False)

origin = np.array([0, 0, 0])
_obs_lens_points = np.array(
    [[1.5, -0.3, 0],
     [0, 1.5, 0.7]], dtype=float
)
obs_lens_points = _obs_lens_points.copy()
lensing_pln_normal = np.cross(*obs_lens_points)
obs_ray = pv.Line(origin, obs_lens_points[0])
lens_ray = pv.Line(origin, obs_lens_points[1])
pvplt.add_mesh(obs_ray, color='dodgerblue', line_width=3, render_lines_as_tubes=False)
pvplt.add_mesh(lens_ray, color='black', line_width=3, render_lines_as_tubes=False)

theta = np.pi / 4
rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
image_obs = rot_mat @ obs_lens_points[0]
image_obs /= np.linalg.norm(image_obs)

image_pos = pv.Sphere(radius=0.1, center=image_obs)
pvplt.add_mesh(image_pos, color='orange')
lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2).triangulate()
pvplt.add_mesh(lensing_pln, color='grey', opacity=0.7, show_edges=False)
intersection = lensing_pln.intersection(unit_sphere, split_first=False, split_second=True)[0]
pvplt.add_mesh(intersection, color='orange', opacity=0.5, line_width=3, show_edges=True)

# Compute theta, phi hats
theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
theta_h_rot = rot_mat @ theta_h
phi_h_rot = rot_mat @ phi_h
theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

arrow_list = []
for centre, direction, color in [
    (obs_lens_points[0], theta_h, 'grey'),
    (obs_lens_points[0], phi_h, 'grey'),
    (image_obs, theta_prime_h, 'grey'),
    (image_obs, phi_prime_h, 'grey'),
    (image_obs, theta_h_rot, 'red'),
    (image_obs, phi_h_rot, 'red'),
]:
    arrow = pv.Arrow(centre, direction)
    pvplt.add_mesh(arrow, color=color, show_edges=False)
    arrow_list.append(arrow)

def update_objects():
    global obs_lens_points
    global lensing_pln_normal
    global lensing_pln
    global intersection
    global theta
    global image_pos
    global arrow_list
    
    _lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2).triangulate()
    lensing_pln.copy_from(_lensing_pln)
    _intersection = _lensing_pln.intersection(unit_sphere, split_first=False, split_second=True)[0]
    intersection.copy_from(_intersection)
    rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
    image_obs = rot_mat @ obs_lens_points[0]
    image_obs /= np.linalg.norm(image_obs)
    _image_pos = pv.Sphere(radius=0.1, center=image_obs)
    image_pos.copy_from(_image_pos)

    # Compute theta, phi hats
    # vectors.mapper.center = obs_lens_points[0]
    theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
    theta_h_rot = rot_mat @ theta_h
    phi_h_rot = rot_mat @ phi_h
    theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

    for arrow, (centre, direction) in zip(arrow_list, [
        (obs_lens_points[0], theta_h),
        (obs_lens_points[0], phi_h),
        (image_obs, theta_prime_h),
        (image_obs, phi_prime_h),
        (image_obs, theta_h_rot),
        (image_obs, phi_h_rot),
    ]):
        _arrow = pv.Arrow(centre, direction, scale=0.2)
        arrow.copy_from(_arrow)

    return np.arccos(np.dot(theta_h_rot, theta_prime_h))
    

def callback(point, i):
    global obs_lens_points
    global lensing_pln
    global lensing_pln_normal

    _obs_lens_points[i] = point
    obs_lens_points = _obs_lens_points.copy()
    norms = np.linalg.norm(_obs_lens_points, axis=1)
    obs_lens_points /= norms[:, None]
    lensing_pln_normal = np.cross(*obs_lens_points)
    lensing_pln_normal /= np.linalg.norm(lensing_pln_normal)

    _obs_ray = pv.Line(origin, _obs_lens_points[0])
    _lens_ray = pv.Line(origin, _obs_lens_points[1])
    obs_ray.copy_from(_obs_ray)
    lens_ray.copy_from(_lens_ray)
    update_objects()

def update_angle(val):
    global theta
    theta = val
    return update_objects()

# pvplt.iren.initialize()
pvplt.add_sphere_widget(callback, center=obs_lens_points, color=['dodgerblue', 'black'], radius=0.07)

# Open a gif
# pvplt.open_gif("rotate.gif")
pvplt.open_movie('rotate.mp4', framerate=30, quality=6)
# pvplt.camera.zoom('tight')
pvplt.camera.position = np.array([-2, 1.5, 2]) * 1.5

n_frame = 300
for alpha in np.linspace(0, 2 * np.pi, n_frame+1)[:n_frame]:
    # print(alpha)
    theta = update_angle(alpha)
    pvplt.add_text(fr'misalignment: {theta:.3f} rad', name = 'theta')
    pvplt.write_frame()
    pvplt.remove_actor('theta')

# Closes and finalizes movie
pvplt.close()